In [10]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import Button
from ultralytics import SAM
import torch

In [11]:
# =========================
# Device
# =========================
device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Using device: {device}")

Using device: cuda


In [12]:
sam2_model = SAM("sam_b.pt")

In [13]:
def find_diameter(contours):
    max_distance = 0
    # Iterate over each contour
    for contour in contours:
        # Iterate over each pair of points in the contour
        for i in range(len(contour)):
            for j in range(i + 1, len(contour)):
                pt1 = contour[i][0]
                pt2 = contour[j][0]
                distance = np.sqrt((pt1[0] - pt2[0]) ** 2 + (pt1[1] - pt2[1]) ** 2)
                if distance > max_distance:
                    max_distance = distance
    return max_distance

In [14]:
folder_path = os.path.join("..", "..", "..", "sample_data", "clipped_spread_seeds_lab")

In [15]:
# folder_path = r'/mnt/c/Users/mabdullahbari/OneDrive - Kansas State University\Felderhoff_Lab_Data\Bari_Choton_Joint\Bound_Box_Area_Cropping\Ground_Truth_Panicle\Ideotype_Panicle_Cropping\test_cropped'
images = sorted([
    os.path.join(folder_path, f)
    for f in os.listdir(folder_path)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
])
image_names = [os.path.basename(p) for p in images]
# Read the file with pixel count information
df_ppi = pd.read_csv('manual_pixel_spread_seed.csv')
df_seed_area = pd.DataFrame(columns=[
    "image_name", "dia_pix", "dia_mm", "area_mm2",
    "avg_dia_mm", "avg_area_mm2",
    "height_pix", "height_mm",
    "width_pix", "width_mm",
    "area_rec_mm2", "avg_area_rec"
])
df_ppi

,id,Plot,Pedigree,Rep,Image_Plot,Image,Image_id,Rep.1,Length,Width,Area_Pixel,Area_mm2,pixels_per_mm
0,2005-SC964,2005,SC964,1,2005,2005_1,2005_2005_1,1,622,303,188466,1250,12.44
1,2009-RTx430BL,2009,RTx430BL,1,2009,2009_1,2009_2009_1,1,867,424,367608,1250,17.34
2,2013-Tx7000,2013,Tx7000,1,2013,2013_1,2013_2013_1,1,778,380,295640,1250,15.56
3,2017-RLBK1,2017,RLBK1,1,2017,2017_1,2017_2017_1,1,930,461,428730,1250,18.60
4,2021-SC326-6,2021,SC326-6,1,2021,2021_1,2021_2021_1,1,794,400,317600,1250,15.88
...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,7057-SC348,7057,SC348,3,3021,3021_3,3021_3021_3,3,695,354,246030,1250,13.90
103,7061-2219-3,7061,2219-3_Stg2NIL,3,3025,3025_3,3025_3025_3,3,789,404,318756,1250,15.78
104,7065-SN72_PINK_KAFIR,7065,SN72_PINK_KAFIR,3,4005,4005_3,4005_4005_3,3,810,388,314280,1250,16.20
105,7069-2290-19,7069,2290-19_Stg3NIL,3,6025,6025_3,6025_6025_3,3,680,328,223040,1250,13.60


## Main code block

In [16]:
# =========================
# Image Viewer
# =========================
class ImageViewer:
    def __init__(self, images):
        self.images = images
        self.image_names = [os.path.basename(p) for p in images]
        self.idx = 0

        self.diameters = []
        self.heights = []
        self.widths = []

        self.cid = None

        # ---- Fixed window size: 1280x720 ----
        self.fig, self.ax = plt.subplots(figsize=(12.8, 7.2), dpi=100)
        self.ax.axis("off")

        self.im = self.ax.imshow(plt.imread(self.images[self.idx]))
        self.ax.set_title(self.image_names[self.idx])

        self._buttons()
        self._enable_click()

        plt.show()

    # -------------------------
    # Buttons
    # -------------------------
    def _buttons(self):
        w, h = 0.12, 0.04

        self.bprev = Button(plt.axes([0.01, 0.01, w, h]), "Previous")
        self.bnext = Button(plt.axes([0.14, 0.01, w, h]), "Next")
        self.bclear = Button(plt.axes([0.27, 0.01, w, h]), "Clear")
        self.bsave = Button(plt.axes([0.40, 0.01, w, h]), "Save CSV")

        self.bprev.on_clicked(self.prev)
        self.bnext.on_clicked(self.next)
        self.bclear.on_clicked(self.clear)
        self.bsave.on_clicked(self.save_csv)

    # -------------------------
    # Navigation
    # -------------------------
    def update(self):
        self.ax.clear()
        self.ax.axis("off")
        self.ax.imshow(plt.imread(self.images[self.idx]))
        self.ax.set_title(self.image_names[self.idx])
        self.fig.canvas.draw()

    def prev(self, _):
        self.clear(None)
        self.idx = max(0, self.idx - 1)
        self.update()

    def next(self, _):
        self.save_metrics()
        self.clear(None)
        self.idx = min(len(self.images) - 1, self.idx + 1)
        self.update()

    # -------------------------
    # Click Segmentation
    # -------------------------
    def _enable_click(self):
        self.cid = self.fig.canvas.mpl_connect("button_press_event", self.onclick)

    def onclick(self, event):
        if event.inaxes != self.ax:
            return

        x, y = int(event.xdata), int(event.ydata)

        results = sam2_model(
            self.images[self.idx],
            points=np.array([[x, y]]),
            labels=np.array([1])
        )

        mask = results[0].masks.data[0].cpu().numpy().astype(np.uint8)
        mask *= 255

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        dia = find_diameter(contours)

        rows, cols = np.where(mask > 0)
        h = cols.max() - cols.min()
        w = rows.max() - rows.min()

        self.diameters.append(dia)
        self.heights.append(max(h, w))
        self.widths.append(min(h, w))

        self.ax.contour(mask, colors="lime", linewidths=1)
        self.fig.canvas.draw()

    # -------------------------
    # Data Handling
    # -------------------------
    def save_metrics(self):
        global df_seed_area

        name = self.image_names[self.idx].split(".")[0]
        ppm = df_ppi.loc[df_ppi.Image_id == name, "pixels_per_mm"].values[0]

        dia_mm = np.array(self.diameters) / ppm
        area_mm = np.pi * (dia_mm / 2) ** 2

        rec_area = (np.array(self.heights) / ppm) * (np.array(self.widths) / ppm)

        for i in range(len(dia_mm)):
            df_seed_area.loc[len(df_seed_area)] = [
                name,
                self.diameters[i],
                dia_mm[i],
                area_mm[i],
                None, None,
                self.heights[i], self.heights[i] / ppm,
                self.widths[i], self.widths[i] / ppm,
                rec_area[i], None
            ]

        df_seed_area.loc[len(df_seed_area)] = [
            name, None, None, None,
            dia_mm.mean(), area_mm.mean(),
            None, None, None, None,
            None, rec_area.mean()
        ]

    def save_csv(self, _):
        self.save_metrics()
        df_seed_area.to_csv("digital_seed_attributes_clean.csv", index=False)
        print("Saved digital_seed_attributes_clean.csv")

    def clear(self, _):
        self.diameters.clear()
        self.heights.clear()
        self.widths.clear()
        self.update()

In [17]:
# Run
# =========================
viewer = ImageViewer(images)

%matplotlib tk

In [18]:
df_seed_area

,image_name,dia_pix,dia_mm,area_mm2,avg_dia_mm,avg_area_mm2,height_pix,height_mm,width_pix,width_mm,area_rec_mm2,avg_area_rec
0,2005_2005_1,50.447993,4.055305,12.916264,NaN,NaN,49,3.938907,45,3.617363,14.248457,NaN
1,2005_2005_1,49.091751,3.946282,12.231118,NaN,NaN,47,3.778135,44,3.536977,13.363179,NaN
2,2005_2005_1,64.637450,5.195937,21.203988,NaN,NaN,54,4.340836,37,2.974277,12.910847,NaN
3,2005_2005_1,53.600373,4.308712,14.580914,NaN,NaN,49,3.938907,45,3.617363,14.248457,NaN
4,2005_2005_1,44.045431,3.540630,9.845796,NaN,NaN,44,3.536977,38,3.054662,10.804272,NaN
5,2005_2005_1,51.662365,4.152923,13.545583,NaN,NaN,50,4.019293,45,3.617363,14.539242,NaN
6,2005_2005_1,52.801515,4.244495,14.149526,NaN,NaN,51,4.099678,49,3.938907,16.148251,NaN
7,2005_2005_1,53.084838,4.267270,14.301781,NaN,NaN,53,4.260450,46,3.697749,15.754076,NaN
8,2005_2005_1,53.823787,4.326671,14.702718,NaN,NaN,52,4.180064,46,3.697749,15.456829,NaN
9,2005_2005_1,NaN,NaN,NaN,4.226469,14.164188,None,NaN,None,NaN,NaN,14.163734
